# Q3 - Semantic Candidate Generation (Embeddings)

Loads the article embeddings computed on Kaggle (see
`src/compute_embeddings_kaggle.ipynb`), retrieves top-K candidates per user
via mean-pooled click history and brute-force cosine similarity, reports
recall@K, and compares against Q2's BM25 results per SPEC.md's Q3 section.

Run top-to-bottom (or via `python embedding_retrieval.py`) to rebuild
`data/processed/{dataset}/embedding_topk.parquet` and `embedding_metrics.json`.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json

import numpy as np
import pandas as pd

from cs4406m26_assignment1c1.embeddings import mean_pool, batched_top_k


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"could not locate {marker} above {Path.cwd()}")


ROOT = find_repo_root()
DATA_DIR = ROOT / "data" / "processed"
DATASETS = ["ebnerd", "mind"]

RECENT_N_CLICKS = 20
CANDIDATE_K_VALUES = [50, 100, 200]
TOPK_MAX = max(CANDIDATE_K_VALUES)
BATCH_SIZE = 2000
EMBEDDING_MODEL_NAME = "paraphrase-xlm-r-multilingual-v1"  # computed on Kaggle, see src/compute_embeddings_kaggle.ipynb

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

feature_store = {
    name: {
        "articles": pd.read_parquet(DATA_DIR / name / "articles.parquet"),
        "behaviors": pd.read_parquet(DATA_DIR / name / "behaviors.parquet"),
        "history": pd.read_parquet(DATA_DIR / name / "history.parquet"),
    }
    for name in DATASETS
}

embedding_paths = {name: DATA_DIR / name / "article_embeddings.parquet" for name in DATASETS}
missing = [str(p) for p in embedding_paths.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"missing article_embeddings.parquet: {missing}. "
        "Run src/compute_embeddings_kaggle.ipynb on Kaggle first and place the "
        "downloaded files at these paths (see README.md)."
    )

embeddings_raw = {name: pd.read_parquet(embedding_paths[name]) for name in DATASETS}
{name: {table: df.shape for table, df in tables.items()} for name, tables in feature_store.items()}

{'ebnerd': {'articles': (11777, 8),
  'behaviors': (50080, 8),
  'history': (1935, 6)},
 'mind': {'articles': (65238, 8),
  'behaviors': (230117, 8),
  'history': (94057, 6)}}

## Build corpus embedding matrices

Reindexed to match `articles.parquet`'s `article_id` order explicitly
(not assumed from `article_embeddings.parquet`'s row order), so downstream
`doc_ids`/`matrix` alignment is correct regardless of how the Kaggle notebook
wrote its output.

In [2]:
corpus = {}
for name in DATASETS:
    articles = feature_store[name]["articles"]
    emb_lookup = dict(zip(embeddings_raw[name]["article_id"], embeddings_raw[name]["embedding"].apply(np.asarray)))
    doc_ids = articles["article_id"].to_numpy()

    missing_ids = set(doc_ids) - set(emb_lookup)
    if missing_ids:
        raise ValueError(f"{name}: {len(missing_ids)} articles have no embedding")

    matrix = np.stack([emb_lookup[aid] for aid in doc_ids]).astype(np.float32)
    corpus[name] = {"doc_ids": doc_ids, "matrix": matrix, "embedding_lookup": emb_lookup}

{name: c["matrix"].shape for name, c in corpus.items()}

{'ebnerd': (11777, 768), 'mind': (65238, 768)}

In [3]:
def test_corpus_embeddings_aligned():
    for name in DATASETS:
        articles = feature_store[name]["articles"]
        c = corpus[name]
        assert c["matrix"].shape[0] == len(articles)
        assert (c["doc_ids"] == articles["article_id"].to_numpy()).all()
        assert not np.isnan(c["matrix"]).any()
        assert set(embeddings_raw[name]["article_id"]) == set(articles["article_id"])


test_corpus_embeddings_aligned()
print("ok: corpus embedding matrices aligned with articles.parquet, no NaNs")

ok: corpus embedding matrices aligned with articles.parquet, no NaNs


## Mean-pooling user representation

Same recency window as Q2's BM25 query construction (`sequence[-20:]`, the
*last* N clicks) -- kept identical so the lexical-vs-semantic comparison
holds the input click window constant.

In [4]:
def build_user_query_vector(article_id_sequence, embedding_lookup: dict, recent_n: int = RECENT_N_CLICKS):
    recent_ids = list(article_id_sequence)[-recent_n:]
    return mean_pool(recent_ids, embedding_lookup)

In [5]:
def test_build_user_query_vector():
    assert build_user_query_vector([], {}) is None

    lookup = {"a": np.array([1.0, 0.0]), "b": np.array([0.0, 1.0]), "c": np.array([2.0, 0.0])}
    # recent_n=1 should only see the *last* id ("c"), not "a"
    vec = build_user_query_vector(["a", "b", "c"], lookup, recent_n=1)
    assert np.allclose(vec, [2.0, 0.0])

    vec_all = build_user_query_vector(["a", "b", "c"], lookup, recent_n=10)
    assert np.allclose(vec_all, np.mean([lookup["a"], lookup["b"], lookup["c"]], axis=0))


test_build_user_query_vector()
print("ok: mean-pooling takes the most recent N clicks (last N, not first N)")

ok: mean-pooling takes the most recent N clicks (last N, not first N)


## Top-K retrieval smoke test

`batched_top_k` is exact within one call (see its docstring for a documented
float32 caveat about comparing *separate* calls with different batch
shapes) -- this test only ever compares results from a single, consistently-shaped
call, which is also how the real per-user retrieval cache below uses it.

In [6]:
def test_batched_top_k():
    name = "mind" if "mind" in DATASETS else DATASETS[0]
    c = corpus[name]

    # self-similarity: a 'user' whose only click is doc 0 should retrieve doc 0 first, score ~= 1.0
    query = c["matrix"][0:1]
    top5 = batched_top_k(query, c["matrix"], c["doc_ids"], k=5, batch_size=BATCH_SIZE)[0]
    assert top5[0][0] == c["doc_ids"][0]
    assert abs(top5[0][1] - 1.0) < 1e-3

    scores = [s for _, s in top5]
    assert scores == sorted(scores, reverse=True)

    # nesting invariant -- both calls use the same single-query batch shape (see docstring caveat)
    top200 = batched_top_k(query, c["matrix"], c["doc_ids"], k=200, batch_size=BATCH_SIZE)[0]
    top50 = batched_top_k(query, c["matrix"], c["doc_ids"], k=50, batch_size=BATCH_SIZE)[0]
    assert top200[:50] == top50

    assert batched_top_k(query, c["matrix"], c["doc_ids"], k=5, batch_size=BATCH_SIZE) == \
           batched_top_k(query, c["matrix"], c["doc_ids"], k=5, batch_size=1)


test_batched_top_k()
print("ok: top-K retrieval is sorted, self-similarity sane, and nested within a call")

ok: top-K retrieval is sorted, self-similarity sane, and nested within a call


## Per-user retrieval cache (val/test users only)

Same cold-start definition Q2 uses (empty `article_id_sequence`) and the
same val/test-only scope -- recomputed here (separate notebook process) but
cross-checked below against Q2's already-persisted `bm25_metrics.json`
counts, which must match exactly since both come from the same `behaviors`/
`history` tables.

In [7]:
user_topk = {}
coldstart_users = {}
for name in DATASETS:
    behaviors = feature_store[name]["behaviors"]
    history = feature_store[name]["history"]
    eval_user_ids = set(behaviors.loc[behaviors["split"].isin(["val", "test"]), "user_id"])
    history_eval = history[history["user_id"].isin(eval_user_ids)]
    lookup = corpus[name]["embedding_lookup"]

    query_user_ids, query_vectors = [], []
    coldstart = set()
    for user_id, article_id_sequence in zip(history_eval["user_id"], history_eval["article_id_sequence"]):
        vec = build_user_query_vector(article_id_sequence, lookup)
        if vec is None:
            coldstart.add(user_id)
            continue
        query_user_ids.append(user_id)
        query_vectors.append(vec)

    query_matrix = np.stack(query_vectors).astype(np.float32) if query_vectors else np.zeros((0, corpus[name]["matrix"].shape[1]))
    topk_results = batched_top_k(query_matrix, corpus[name]["matrix"], corpus[name]["doc_ids"], k=TOPK_MAX, batch_size=BATCH_SIZE)

    user_topk[name] = dict(zip(query_user_ids, topk_results))
    coldstart_users[name] = coldstart

{name: {"retrieved": len(user_topk[name]), "coldstart": len(coldstart_users[name])} for name in DATASETS}

{'ebnerd': {'retrieved': 1619, 'coldstart': 0},
 'mind': {'retrieved': 65173, 'coldstart': 1770}}

In [8]:
def test_user_retrieval_cache():
    for name in DATASETS:
        behaviors = feature_store[name]["behaviors"]
        eval_user_ids = set(behaviors.loc[behaviors["split"].isin(["val", "test"]), "user_id"])
        assert set(user_topk[name]).issubset(eval_user_ids)
        assert set(coldstart_users[name]).issubset(eval_user_ids)
        assert set(user_topk[name]) | set(coldstart_users[name]) == eval_user_ids

        for aid_scores in list(user_topk[name].values())[:5]:
            assert 1 <= len(aid_scores) <= TOPK_MAX

        # cross-check against Q2's already-persisted cold-start counts -- same
        # users, same criterion, computed independently in a separate notebook
        bm25_metrics_path = DATA_DIR / name / "bm25_metrics.json"
        if bm25_metrics_path.exists():
            bm25_metrics = json.loads(bm25_metrics_path.read_text())
            for split in ["val", "test"]:
                expected_coldstart_impressions = bm25_metrics["n_impressions"][split]["excluded_coldstart"]
                actual_coldstart_impressions = behaviors[
                    (behaviors["split"] == split) & (behaviors["user_id"].isin(coldstart_users[name]))
                ].shape[0]
                assert actual_coldstart_impressions == expected_coldstart_impressions, (
                    f"{name}/{split}: cold-start impression count diverged from Q2's bm25_metrics.json"
                )


test_user_retrieval_cache()
print("ok: retrieval cache covers exactly the val/test user population, cold-start matches Q2's numbers")

ok: retrieval cache covers exactly the val/test user population, cold-start matches Q2's numbers


## Recall@K evaluation

Same fractional multi-relevant definition as Q2 (`article_ids_clicked` is
not always singleton) -- retrieval-method-independent, reused directly.

In [9]:
def evaluate_recall(dataset: str, split: str, k_values: list[int]) -> dict:
    behaviors = feature_store[dataset]["behaviors"]
    split_behaviors = behaviors[behaviors["split"] == split]
    topk = user_topk[dataset]
    coldstart = coldstart_users[dataset]

    recalls = {k: [] for k in k_values}
    n_excluded = 0

    for user_id, clicked in zip(split_behaviors["user_id"], split_behaviors["article_ids_clicked"]):
        if user_id in coldstart:
            n_excluded += 1
            continue
        clicked = set(clicked)
        candidate_ids = [aid for aid, _ in topk[user_id]]
        for k in k_values:
            top_k_ids = set(candidate_ids[:k])
            recalls[k].append(len(clicked & top_k_ids) / len(clicked))

    n_evaluated = len(split_behaviors) - n_excluded
    return {
        "recall_at_k": {k: (sum(v) / len(v) if v else 0.0) for k, v in recalls.items()},
        "n_total": len(split_behaviors),
        "n_evaluated": n_evaluated,
        "n_excluded_coldstart": n_excluded,
    }


metrics = {name: {split: evaluate_recall(name, split, CANDIDATE_K_VALUES) for split in ["val", "test"]} for name in DATASETS}
metrics

{'ebnerd': {'val': {'recall_at_k': {50: 0.008807985907222548,
    100: 0.014679976512037582,
    200: 0.028479154433352905},
   'n_total': 3406,
   'n_evaluated': 3406,
   'n_excluded_coldstart': 0},
  'test': {'recall_at_k': {50: 0.005324183625177473,
    100: 0.011693484776778671,
    200: 0.026062207498553928},
   'n_total': 25356,
   'n_evaluated': 25356,
   'n_excluded_coldstart': 0}},
 'mind': {'val': {'recall_at_k': {50: 0.008918132195134026,
    100: 0.014341567496018644,
    200: 0.023930080720961165},
   'n_total': 30270,
   'n_evaluated': 29498,
   'n_excluded_coldstart': 772},
  'test': {'recall_at_k': {50: 0.006506347717334776,
    100: 0.0109387725950781,
    200: 0.019198809599934523},
   'n_total': 73152,
   'n_evaluated': 70938,
   'n_excluded_coldstart': 2214}}}

In [10]:
def test_recall_at_k_monotonic():
    for name in DATASETS:
        for split in ["val", "test"]:
            m = metrics[name][split]
            r = m["recall_at_k"]
            assert r[50] <= r[100] + 1e-12 <= r[200] + 1e-12
            assert all(0.0 <= v <= 1.0 for v in r.values())
            assert m["n_evaluated"] + m["n_excluded_coldstart"] == m["n_total"]


test_recall_at_k_monotonic()
print("ok: recall@K is monotonic in K, bounded in [0,1], and impression counts are consistent")

ok: recall@K is monotonic in K, bounded in [0,1], and impression counts are consistent


## Lexical vs. semantic comparison

Reads both `bm25_metrics.json` (Q2) and this notebook's `metrics` (embeddings)
to compare recall@K side by side, per dataset/split.

In [11]:
comparison_rows = []
for name in DATASETS:
    bm25_metrics_path = DATA_DIR / name / "bm25_metrics.json"
    bm25_metrics = json.loads(bm25_metrics_path.read_text()) if bm25_metrics_path.exists() else None
    for split in ["val", "test"]:
        for k in CANDIDATE_K_VALUES:
            row = {
                "dataset": name,
                "split": split,
                "k": k,
                "embedding_recall": metrics[name][split]["recall_at_k"][k],
                "bm25_recall": bm25_metrics["recall_at_k"][split][str(k)] if bm25_metrics else None,
            }
            comparison_rows.append(row)

comparison_table = pd.DataFrame(comparison_rows)
comparison_table

,dataset,split,k,embedding_recall,bm25_recall
0,ebnerd,val,50,0.008808,0.008808
1,ebnerd,val,100,0.014680,0.019378
2,ebnerd,val,200,0.028479,0.035085
3,ebnerd,test,50,0.005324,0.006981
4,ebnerd,test,100,0.011693,0.014612
5,ebnerd,test,200,0.026062,0.028523
6,mind,val,50,0.008918,0.011379
7,mind,val,100,0.014342,0.021970
8,mind,val,200,0.023930,0.034001
9,mind,test,50,0.006506,0.004478


In [12]:
def test_comparison_table():
    assert len(comparison_table) == len(DATASETS) * 2 * len(CANDIDATE_K_VALUES)
    assert set(comparison_table["dataset"]) == set(DATASETS)
    if comparison_table["bm25_recall"].notna().all():
        for name in DATASETS:
            bm25_metrics = json.loads((DATA_DIR / name / "bm25_metrics.json").read_text())
            for split in ["val", "test"]:
                for k in CANDIDATE_K_VALUES:
                    row = comparison_table[
                        (comparison_table["dataset"] == name)
                        & (comparison_table["split"] == split)
                        & (comparison_table["k"] == k)
                    ].iloc[0]
                    assert abs(row["bm25_recall"] - bm25_metrics["recall_at_k"][split][str(k)]) < 1e-12


test_comparison_table()
print("ok: lexical-vs-semantic comparison table matches both persisted metrics files")

ok: lexical-vs-semantic comparison table matches both persisted metrics files


## Persist embedding outputs

Schema-parallel to Q2's `bm25_topk.parquet`/`bm25_metrics.json` (see
`SPEC.md` Q3 section 6).

In [13]:
def write_embedding_outputs(dataset: str) -> Path:
    out_dir = DATA_DIR / dataset
    rows = [
        {
            "user_id": user_id,
            "dataset": dataset,
            "n_retrieved": len(aid_scores),
            "retrieved_article_ids": [aid for aid, _ in aid_scores],
            "retrieved_scores": [float(s) for _, s in aid_scores],
        }
        for user_id, aid_scores in user_topk[dataset].items()
    ]
    pd.DataFrame(rows).to_parquet(out_dir / "embedding_topk.parquet", index=False)

    embedding_metrics = {
        "schema_version": 1,
        "build_timestamp": datetime.now(timezone.utc).isoformat(),
        "hyperparameters": {
            "embedding_method": EMBEDDING_MODEL_NAME,
            "embedding_dim": int(corpus[dataset]["matrix"].shape[1]),
            "recent_n_clicks": RECENT_N_CLICKS,
            "topk_max": TOPK_MAX,
        },
        "recall_at_k": {split: metrics[dataset][split]["recall_at_k"] for split in ["val", "test"]},
        "n_impressions": {
            split: {
                "total": metrics[dataset][split]["n_total"],
                "evaluated": metrics[dataset][split]["n_evaluated"],
                "excluded_coldstart": metrics[dataset][split]["n_excluded_coldstart"],
            }
            for split in ["val", "test"]
        },
        "scope": "val_test_users_only",
    }
    (out_dir / "embedding_metrics.json").write_text(json.dumps(embedding_metrics, indent=2))
    return out_dir


embedding_out_dirs = {name: write_embedding_outputs(name) for name in DATASETS}
embedding_out_dirs

{'ebnerd': WindowsPath('C:/Users/HP/cs4406m26-assignment1c1/data/processed/ebnerd'),
 'mind': WindowsPath('C:/Users/HP/cs4406m26-assignment1c1/data/processed/mind')}

In [14]:
def test_embedding_outputs_roundtrip():
    for name in DATASETS:
        out_dir = embedding_out_dirs[name]
        topk_path = out_dir / "embedding_topk.parquet"
        metrics_path = out_dir / "embedding_metrics.json"
        assert topk_path.exists() and metrics_path.exists()

        reloaded_topk = pd.read_parquet(topk_path)
        assert set(reloaded_topk["user_id"]) == set(user_topk[name])
        for _, row in reloaded_topk.head(20).iterrows():
            assert len(row["retrieved_article_ids"]) == row["n_retrieved"]
            assert len(row["retrieved_scores"]) == row["n_retrieved"]

        reloaded_metrics = json.loads(metrics_path.read_text())
        for split in ["val", "test"]:
            for k in CANDIDATE_K_VALUES:
                expected = metrics[name][split]["recall_at_k"][k]
                actual = reloaded_metrics["recall_at_k"][split][str(k)]
                assert abs(expected - actual) < 1e-12


test_embedding_outputs_roundtrip()
print("ok: embedding_topk.parquet and embedding_metrics.json round-trip correctly for both datasets")

ok: embedding_topk.parquet and embedding_metrics.json round-trip correctly for both datasets
